In [1]:
import sys
sys.path.append('../')

import scqubits as scq
import pandas as pd
import qutip as qt
import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
import cmath
from tqdm import tqdm
from matplotlib.colors import LogNorm
import datetime
import pytz
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import itertools
import scipy.sparse as ssp
from sympy import symbols
import scipy as sp
import utils_2Q_gate_zp as ut
import os
from datetime import datetime
from multiprocessing import Pool

In [22]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[[0], :]

num_cpus, n_job = 4, 1*len(params)
logi_state = [0, 2]
folder = '../../data/3ncut_one_zeropi/'
if drive_0:
    evals = 2*np.pi* scq.read(folder + f'zeropi_0_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_0_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_0_n_phi_truc=1000_3ncut.h5').matrixelem_table
else:
    evals = 2*np.pi* scq.read(folder + f'zeropi_1_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_1_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_1_n_phi_truc=1000_3ncut.h5').matrixelem_table
evals = evals - evals[0]
H0 = qt.Qobj(np.diag(evals))
if drive_phi:
    w_trans_1 = evals[9] - evals[0]
    w_trans_2 = evals[9] - evals[2]
    drive_term = n_phi
if drive_theta:
    w_trans_1 = evals[7] - evals[0]
    w_trans_2 = evals[7] - evals[2]
    drive_term = n_theta

############################################################
thresh = 0.01
hspace_charge = [0, 2]  # Start with the ground and first excited states
for s in hspace_charge:
    for i in range(truc):
        if np.abs(drive_term[s, i] / (2 * np.pi)) > thresh and i not in hspace_charge:
            hspace_charge.append(i)
hspace_charge.sort()
############################################################
# hspace_charge = np.arange(truc).tolist()

############################################################
hspace_len = len(hspace_charge)
logi_idx = [hspace_charge.index(s) for s in logi_state]
H0_truc = ut.truncate_2(H0, hspace_charge)
drive_truc = ut.truncate_2(drive_term, hspace_charge)
H_qbt_drive = [H0_truc, [drive_truc, ut.drive_gauss_A],
                        [drive_truc, ut.drive_gauss_B],]
############################################################
print('params =')
for para in params:
    print(para.tolist(), ',')
print('hspace_len=', hspace_len)
print(' hspace_charge = [')
for i in range(0, len(hspace_charge), 10):
    print(', '.join(map(str, hspace_charge[i:i+10])), ',')
print(']')

params =
[20.000087, 0.249753, 0.216297, 0.43883, 0.490612] ,
hspace_len= 81
 hspace_charge = [
0, 2, 3, 4, 8, 9, 10, 11, 15, 16 ,
18, 19, 20, 23, 24, 25, 28, 31, 33, 35 ,
37, 39, 40, 42, 43, 46, 47, 49, 51, 53 ,
55, 56, 57, 60, 62, 64, 66, 67, 69, 70 ,
73, 76, 77, 79, 81, 83, 85, 86, 88, 91 ,
92, 95, 96, 98, 100, 101, 102, 104, 106, 108 ,
110, 112, 113, 116, 118, 120, 121, 123, 126, 128 ,
130, 132, 133, 136, 137, 139, 141, 143, 145, 146 ,
148 ,
]


### New model - decay to not just $\ket{0}$

In [23]:
folder_1 = 'data/data_gamma_'
folder_2 = 'theta.txt' if drive_theta else 'phi_truc200.txt'
gamma_new = pd.read_csv(folder_1 + folder_2)
gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

if drive_theta:
    Gamma = gamma_decay_other / (np.abs(n_theta[4,7])**2)
else:
    Gamma = gamma_decay_other / (np.abs(n_phi[4,9])**2)
gamma_decay_new = Gamma* np.abs(drive_truc.full())**2    
gamma_dephase_new = gamma_dephase_new *50 /t1_other
jump_t1   = []
jump_tphi = []
for i in range(1,hspace_len):
    for j in range(0,i):
        jump_t1.append( np.sqrt(gamma_decay_new[i,j]) * qt.basis(hspace_len,j) * qt.basis(hspace_len,i).dag() )
    jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )
print('np.shape(jump_t1)=',  np.shape(jump_t1), '; np.shape(jump_tphi)=',  np.shape(jump_tphi)) 

np.shape(jump_t1)= (3240, 81, 81) ; np.shape(jump_tphi)= (80, 81, 81)


### noise simulation

In [24]:
c_op_list = [qt.Qobj(np.zeros((truc, truc)))]
c_op_list = []
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_ideal = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_ideal = [')
for i in range(0, len(f_ideal), 4):
    print(', '.join(map(str, f_ideal[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))

############################################################
c_op_list = jump_t1 + jump_tphi
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_noise = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, f_noise[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))


f_ideal = [
-0.483856699188308 ,
]
Current Mountain Time: 2025-06-13 12:38:04.109427-06:00

f_noise = [
-0.4836847114333318 ,
]
Current Mountain Time: 2025-06-13 12:39:27.444467-06:00


In [25]:
# print(np.shape(n_theta), np.shape(n_phi), np.shape(evals))


In [26]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

### Old model - decay to $\ket{0}$

In [ ]:
# ### old model
# idx_2 = 2 if drive_theta else 1

# hspace_len = len(hspace_charge)
# if drive_theta:
#     gamma_decay_old   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-3)
#     gamma_dephase_old = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-3)
# else:
#     gamma_decay_old   = [0,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-2)
#     gamma_dephase_old = [0,  gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-2)

# folder_1 = 'data/data_gamma_'
# folder_2 = 'theta.txt' if drive_theta else 'phi.txt'
# gamma_new = pd.read_csv(folder_1 + folder_2)

# ### 't1_50us_47', 't1_50us_27', 't1_50us_07'
# ### 'tphi_50us_02', 'tphi_50us_07', 'tphi_1e6'
# gamma_decay_new = gamma_new['t1_50us_47'].to_numpy()
# # gamma_dephase_new = gamma_new['tphi_1e6'].to_numpy()
# gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

# print("gamma_decay_new[2] = ", gamma_decay_new[2], ", gamma_dephase_new[2] = ", gamma_dephase_new[2])
# gamma_decay_new = gamma_decay_new *50 /t1_other
# gamma_dephase_new = gamma_dephase_new *50 /t1_other

# jump_t1   = []
# jump_tphi = []
# for i in range(1,hspace_len):
#     jump_t1.append( np.sqrt(gamma_decay_new[i]) * qt.basis(hspace_len,0) * qt.basis(hspace_len,i).dag() )
#     jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )


In [ ]:
# np.savez("data/data_xgate_theta_collapse_op.npz", arr1=c_op_list)